# Update YARRRML Rules and Generate RDF

In [ ]:
import csv
import json
import os
import re
import subprocess

from rdflib import RDF, XSD, Graph, Literal, Namespace, URIRef

TRAJECTORY_DATA_TTL = '../files/rdf/trajectory_data.ttl'
# Must follow the same scoped directory as the trajectory CSVs. Reading
# another corpus's pois.csv silently yields "Missing POI details" for every
# POI here, which skips the tag-rule tier entirely and leaves them unclassified.
POI_DATA_CSV = os.path.join(
    os.environ.get('TRAJECTO_CSV_DIR', '../files/csv'), 'pois.csv')

In [ ]:
# Get CSV files from files/csv directory
csv_dir = os.environ.get('TRAJECTO_CSV_DIR', '../files/csv')  # set per corpus
csv_files = sorted([f for f in os.listdir(csv_dir) if f.endswith('.csv')])
trajectory_files = [f for f in csv_files if f != 'pois.csv']

print(f"Found {len(trajectory_files)} trajectory files: {trajectory_files}")

In [ ]:
# Read the current rules.yml file
with open('../mappings/rules_template.yml') as f:
    content = f.read()

In [ ]:
# Create trajectory sources in the correct format
# Paths in rules.yml are resolved relative to mappings/. notebooks/ and
# mappings/ are siblings, so csv_dir ("../files/csv...") is correct from both.
trajectory_sources_lines = [f"      - [{csv_dir}/{f}~csv]"
                            for f in trajectory_files]
trajectory_sources_str = '\n'.join(trajectory_sources_lines)

# Find all trajectory sources patterns and replace them
# Pattern matches sources section with one or more trajectory files
pattern = r'(sources:\s*\n)((?:\s*-\s*\[trajectory_\d+\.csv~csv\]\s*\n)+)'
replacement = f'\\1{trajectory_sources_str}\n'
content = re.sub(pattern, replacement, content)

# The template's POI source is hardcoded to ../files/csv; point it at the
# same scoped directory as the trajectories.
content = content.replace('[../files/csv/pois.csv~csv]', f'[{csv_dir}/pois.csv~csv]')

print("Rules updated")

In [ ]:
# Write the updated rules.yml
with open('../mappings/rules.yml', 'w') as f:
    f.write(content)
print("Saved updated rules.yml")

In [ ]:
# Run YARRRML parser
cmd = ['yarrrml-parser', '-i', '../mappings/rules.yml', '-o', '../mappings/rules.rml.ttl']
result = subprocess.run(cmd, capture_output=True, text=True)
print(f"YARRRML parser: {result.returncode}")
if result.stderr:
    print(f"Error: {result.stderr}")

In [ ]:
# Create RDF output directory if it doesn't exist
os.makedirs('../files/rdf', exist_ok=True)

# Change to mappings directory to run RML mapper (for correct relative paths)
os.chdir('../mappings')

# Run RML mapper. The mapper holds every source CSV in memory at once, so the
# heap has to scale with the corpus: the JVM default (6 GB here, a quarter of
# physical RAM) is fine for a few hundred files but dies with OutOfMemoryError
# on ~1,200. TRAJECTO_RML_HEAP overrides it.
rml_heap = os.environ.get('TRAJECTO_RML_HEAP', '16g')
cmd = ['java', f'-Xmx{rml_heap}', '-jar', 'rmlmapper-7.3.3-r374-all.jar',
       '-m', 'rules.rml.ttl',
       '-o', TRAJECTORY_DATA_TTL,
       '-s', 'turtle']
print(f'RML heap: {rml_heap}')
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout:
    print(result.stdout[:500])
if result.returncode != 0 and result.stderr:
    print(f'STDERR: {result.stderr[:1000]}')

print(f"RML mapper: {result.returncode}")
if result.stderr:
    print(f"Error: {result.stderr}")

# Change back to notebooks directory
os.chdir('../notebooks')

if result.returncode != 0:
    raise RuntimeError(
        f'RML mapper failed (exit {result.returncode}). Refusing to continue: '
        f'the next cell would load a stale {TRAJECTORY_DATA_TTL} left by a '
        f'previous corpus and upload it as if it were this one.')

In [ ]:
def transform_poi_interactions(input_file, output_file):
    """
    Transform POI interactions from bracketed list format to individual triples.
    Preserves ALL existing triples including datacron:Point entities.

    Args:
        input_file: Path to input Turtle file
        output_file: Path to output Turtle file
    """
    # Define namespaces
    POI = Namespace("http://w3id.org/trajecto/poi#")
    SPATIAL = Namespace("http://w3id.org/trajecto/spatial#")
    DATACRON = Namespace("http://www.datacron-project.eu/datAcron#")

    # Load the graph
    g = Graph()
    g.parse(input_file, format='turtle')

    # Bind namespaces for prettier output
    g.bind('poi', POI)
    g.bind('spatial', SPATIAL)
    g.bind('datacron', DATACRON)

    # Count important entities before transformation
    point_count_before = sum(1 for _ in g.subjects(RDF.type, DATACRON.Point))
    triples_before = len(g)

    print("Graph loaded:")
    print(f"  - Total triples: {triples_before}")
    print(f"  - datacron:Point entities: {point_count_before}")

    # Find all POI interactions with bracketed lists
    pattern = re.compile(r"pointOfInterest_\['([^']+(?:',\s*'[^']+)*)'\]")

    # Track changes to make
    to_remove = []
    to_add = []

    # Query for all POI interactions
    for subj, pred, obj in g:
        if pred == POI.interactsWith:
            obj_str = str(obj)
            match = pattern.search(obj_str)

            if match:
                # Extract the IDs from the brackets
                values_str = match.group(1)
                # Split by ',' to handle the comma and space between quoted values
                values = [v.strip() for v in values_str.split("', '")]

                # Mark old triple for removal
                to_remove.append((subj, pred, obj))

                # Process each ID-time pair
                for value in values:
                    # Split by '-' to get ID and distance (pairs are
                    # built in notebook 1 as f"{id}-{distance_m}")
                    parts = value.split('-')
                    if len(parts) == 2:
                        id_num = parts[0].strip()
                        distance_m = parts[1].strip()

                        # Create new triple for the POI interaction
                        new_uri = URIRef(f"http://w3id.org/trajecto/poi#pointOfInterest_{id_num}")
                        to_add.append((subj, pred, new_uri))

                        # Create new triple for the position-to-POI distance
                        to_add.append((subj, SPATIAL.hasDistance, Literal(distance_m, datatype=XSD.decimal)))

    # Apply changes - ONLY modify POI interaction triples
    for triple in to_remove:
        g.remove(triple)

    for triple in to_add:
        g.add(triple)

    # Count after transformation
    point_count_after = sum(1 for _ in g.subjects(RDF.type, DATACRON.Point))
    triples_after = len(g)

    # Save the transformed graph
    g.serialize(destination=output_file, format='turtle')

    print("\nTransformation complete!")
    print(f"  - Removed {len(to_remove)} bracketed list triple(s)")
    print(f"  - Added {len(to_add)} individual triple(s)")
    print("\nGraph after transformation:")
    print(f"  - Total triples: {triples_before} → {triples_after} (change: {triples_after - triples_before:+d})")
    print(f"  - datacron:Point entities: {point_count_before} → {point_count_after}")

    if point_count_after < point_count_before:
        print(f"  ⚠️  WARNING: Lost {point_count_before - point_count_after} Point entities during serialization!")
    elif point_count_after == point_count_before and point_count_after > 0:
        print(f"  ✓ All {point_count_after} Point entities preserved")

    print(f"  - Output saved to: {output_file}")

    return g


transform_poi_interactions(TRAJECTORY_DATA_TTL, TRAJECTORY_DATA_TTL)

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

# Thread-safe lock for cache updates
_classification_lock = threading.Lock()


def classify_poi(poi_json: dict) -> list[str]:
    """
    Classify a POI JSON description using the stored GPT-5 Nano prompt.

    Args:
        poi_json (dict): OSM POI description in JSON format

    Returns:
        list[str]: List of ontology class IRIs returned by the model
    """
    response = client.responses.create(
        input=[
            {
                "role": "user",
                "content": f"Classify the following POI:\n{poi_json}"
            }
        ],
        # Link to your stored prompt
        prompt={
            "id": "pmpt_68bb1ab0b8a48194b125851db9b0d8360ac7989211ef8a6a",
            "version": "12"
        }
    )

    # Extract the text output from the model
    output_text = response.output[0].content[0].text.strip()

    # Split lines into a list (since each IRI is expected on its own line)
    return [line.strip() for line in output_text.splitlines() if line.strip()]


def classify_poi_with_id(poi_id: str, poi_tags: str) -> tuple[str, list[str], bool]:
    """
    Classify a single POI and return result with ID.
    Returns: (poi_id, classifications, success)
    """
    try:
        classifications = classify_poi(poi_tags)
        return (poi_id, classifications, True)
    except Exception as e:
        print(f"  Error classifying POI {poi_id}: {e}")
        return (poi_id, [], False)


def classify_pois_batch(pois_to_classify: list[tuple[str, str]], max_workers: int = 8) -> dict[str, list[str]]:
    """
    Classify multiple POIs concurrently.

    Args:
        pois_to_classify: List of (poi_id, poi_tags) tuples
        max_workers: Maximum concurrent API calls

    Returns:
        Dict mapping poi_id to classifications
    """
    results = {}

    if not pois_to_classify:
        return results

    print(f"  Classifying {len(pois_to_classify)} POIs ({max_workers} concurrent)...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(classify_poi_with_id, poi_id, poi_tags): poi_id
            for poi_id, poi_tags in pois_to_classify
        }

        for completed, future in enumerate(as_completed(futures), 1):
            poi_id, classifications, success = future.result()
            if success and classifications:
                results[poi_id] = classifications
            if completed % 10 == 0:
                print(f"  Progress: {completed}/{len(pois_to_classify)}")

    print(f"  Completed: {len(results)} successful classifications")
    return results


# Example usage (smoke test only). Guarded: this fires a live API call on
# every pipeline run, and an API failure here aborts the whole chain before
# the real classification pass -- which needs the LLM for only the POIs the
# tag rules cannot resolve.
if os.environ.get('TRAJECTO_CLASSIFIER_SMOKE_TEST'):
    poi_example = {'int_name': 'Anafiotika', 'name': 'Anafiotika'}
    print(f"Example classification: {classify_poi(poi_example)}")


In [ ]:
from helpers.poi_classifier import classify_by_tags
from helpers.poi_categories import UNDECLARED_CLASS_MAP


def is_valid_poi_uri(uri_str: str) -> bool:
    """
    Check if a POI URI is valid.
    Valid POI URIs should:
    - Start with http:// or https://
    - Contain only valid URI characters
    - Match the expected POI URI pattern (pointOfInterest_<digits>)
    """
    if not uri_str:
        return False

    # Must start with http(s)://
    if not uri_str.startswith(('http://', 'https://')):
        return False

    # Check for invalid characters/patterns
    invalid_patterns = [
        '<|', '|>',           # diff_marker variants
        '|diff_marker|',
        '[', ']',             # brackets (from list artifacts)
        '{', '}',             # braces
        ' ',                  # spaces (invalid in URIs)
        '\n', '\t',           # whitespace
        '"', "'",             # quotes
    ]

    for pattern in invalid_patterns:
        if pattern in uri_str:
            return False

    # Check for valid POI pattern: should end with pointOfInterest_<digits>
    poi_pattern = re.search(r'pointOfInterest_(\d+)$', uri_str)
    return bool(poi_pattern)


def get_poi_id_from_uri(poi_uri):
    """
    Extract POI ID from a URI string.
    Returns None for invalid URIs.
    """
    uri_str = str(poi_uri)

    # Validate URI first
    if not is_valid_poi_uri(uri_str):
        return None

    # Extract the numeric ID
    match = re.search(r'pointOfInterest_(\d+)$', uri_str)
    if match:
        return match.group(1)
    return None


def add_poi_classifications(graph_file, poi_details_csv, output_file=None, cache_file='../files/poi_classification_cache.json', max_workers=8):
    """
    Add POI classifications to the graph based on POI details from CSV.

    Two-tier classifier: deterministic OSM-tag rules run first; only POIs the
    rules can't resolve are sent to the LLM. Both tiers feed the same cache.
    """
    POI = Namespace("http://w3id.org/trajecto/poi#")
    SPATIAL = Namespace("http://w3id.org/trajecto/spatial#")
    TRAJECTO = Namespace("http://w3id.org/trajecto#")

    g = Graph()
    g.parse(graph_file, format='turtle')
    g.bind('poi', POI)
    g.bind('spatial', SPATIAL)
    g.bind('rdf', RDF)

    classification_cache = {}
    if os.path.exists(cache_file):
        try:
            with open(cache_file, encoding='utf-8') as f:
                classification_cache = json.load(f)
            print(f"Loaded {len(classification_cache)} cached POI classifications")
        except Exception as e:
            print(f"Could not load cache file: {e}")
            classification_cache = {}

    poi_details = {}
    with open(poi_details_csv, encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            poi_id = row.get('id')
            if poi_id and 'all_tags' in row:
                poi_details[str(poi_id)] = row['all_tags']

    pois_to_process = {}
    poi_interactions = list(g.subjects(RDF.type, POI.POIInteraction))
    skipped_invalid = []

    print(f"Scanning {len(poi_interactions)} POI interactions...")

    for interaction in poi_interactions:
        related_pois = g.objects(interaction, POI.interactsWith)
        for poi_uri in related_pois:
            uri_str = str(poi_uri)

            if not is_valid_poi_uri(uri_str):
                if uri_str not in skipped_invalid:
                    skipped_invalid.append(uri_str)
                continue

            poi_id = get_poi_id_from_uri(poi_uri)
            if poi_id and poi_id not in pois_to_process:
                pois_to_process[poi_id] = poi_uri

    # Declare trajecto:PointOfInterest supertype for every POI referenced from an interaction.
    # Interactions target both pois.csv entries (RML-typed) and cache-only entries (not RML-typed).
    # Without this, stage-02 SHACL fails since pyshacl does not do subclass inference.
    for _pid, _puri in pois_to_process.items():
        g.add((_puri, RDF.type, TRAJECTO.PointOfInterest))

    print(f"Found {len(pois_to_process)} unique POIs to process")
    if skipped_invalid:
        print(f"  Skipped {len(skipped_invalid)} invalid POI URI(s):")
        for invalid_uri in skipped_invalid[:5]:
            print(f"    - {invalid_uri[:60]}...")
        if len(skipped_invalid) > 5:
            print(f"    ... and {len(skipped_invalid) - 5} more")

    cached_pois = {}
    uncached_pois = []
    missing_details = []

    for poi_id, _poi_uri in pois_to_process.items():
        if poi_id in classification_cache:
            cached_pois[poi_id] = classification_cache[poi_id]
        elif poi_id in poi_details:
            uncached_pois.append((poi_id, poi_details[poi_id]))
        else:
            missing_details.append(poi_id)

    print(f"  Cache hits: {len(cached_pois)}")
    print(f"  Need classification: {len(uncached_pois)}")
    if missing_details:
        print(f"  Missing POI details: {len(missing_details)}")

    # Tier 1: deterministic OSM-tag rules
    rule_classified = {}
    llm_fallback = []
    for poi_id, poi_tags in uncached_pois:
        rule_classes = classify_by_tags(poi_tags)
        if rule_classes:
            rule_classified[poi_id] = rule_classes
        else:
            llm_fallback.append((poi_id, poi_tags))

    if rule_classified:
        print(f"  Rule-based hits: {len(rule_classified)}")
        classification_cache.update(rule_classified)

    # Tier 2: LLM for tags the rules can't resolve
    new_classifications = {}
    if llm_fallback:
        print(f"  LLM fallback: {len(llm_fallback)}")
        new_classifications = classify_pois_batch(llm_fallback, max_workers=max_workers)
        classification_cache.update(new_classifications)

    all_classifications = {**cached_pois, **rule_classified, **new_classifications}

    print("\nAdding classifications to graph...")
    classified_count = 0
    skipped_classifications = 0

    for poi_id, poi_uri in pois_to_process.items():
        if poi_id in all_classifications:
            classifications = all_classifications[poi_id]

            # Ensure every classified POI carries the trajecto:PointOfInterest supertype
            g.add((poi_uri, RDF.type, TRAJECTO.PointOfInterest))

            for classification in classifications:
                try:
                    classification = classification.strip()

                    if not classification or '|' in classification or '<' in classification or '>' in classification:
                        skipped_classifications += 1
                        continue

                    if ':' in classification:
                        prefix, local_name = classification.split(':', 1)
                        if not re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', local_name):
                            skipped_classifications += 1
                            continue
                        if prefix == 'poi':
                            if local_name in UNDECLARED_CLASS_MAP:   # classifier name the module lacks
                                local_name = UNDECLARED_CLASS_MAP[local_name]
                                if local_name is None:
                                    continue
                            class_uri = POI[local_name]
                        else:
                            class_uri = URIRef(f"http://w3id.org/trajecto/{prefix}#{local_name}")
                    else:
                        if not re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', classification):
                            skipped_classifications += 1
                            continue
                        if classification in UNDECLARED_CLASS_MAP:
                            classification = UNDECLARED_CLASS_MAP[classification]
                            if classification is None:
                                continue
                        class_uri = POI[classification]

                    g.add((poi_uri, RDF.type, class_uri))
                except Exception:
                    skipped_classifications += 1

            classified_count += 1

    if skipped_classifications > 0:
        print(f"  Skipped {skipped_classifications} invalid classification(s)")

    try:
        os.makedirs(os.path.dirname(cache_file), exist_ok=True)
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(classification_cache, f, indent=2, ensure_ascii=False)
        print(f"✓ Saved classification cache to: {cache_file}")
    except Exception as e:
        print(f"Warning: Could not save cache file: {e}")

    if output_file:
        try:
            g.serialize(destination=output_file, format='turtle')
            print(f"✓ Updated graph saved to: {output_file}")
        except Exception as e:
            print(f"Warning: Could not save graph: {e}")

    print("\nClassification complete!")
    print(f"- Classified {classified_count} unique POI(s)")
    print(f"  \u2192 From cache: {len(cached_pois)}")
    print(f"  \u2192 From rules: {len(rule_classified)}")
    print(f"  \u2192 From LLM:   {len(new_classifications)}")
    print(f"- Total cached: {len(classification_cache)}")

    return g


add_poi_classifications(
    graph_file=TRAJECTORY_DATA_TTL,
    poi_details_csv=POI_DATA_CSV,
    output_file=TRAJECTORY_DATA_TTL,
    max_workers=8
)


In [ ]:
# Ensure Fuseki datasets exist before uploading
from helpers.fuseki_config import ensure_datasets_exist

if not ensure_datasets_exist():
    raise RuntimeError("Failed to setup Fuseki datasets. Please ensure Fuseki is running.")

In [ ]:
# Validate RDF data against stage-02 SHACL shapes before uploading to Fuseki
from helpers.shacl_validator import validate_ttl_file

report = validate_ttl_file(TRAJECTORY_DATA_TTL, stage='stage-02-base-rdf')

In [ ]:
import requests
from helpers.fuseki_config import fuseki_server, graph_name, upload_ontologies


def upload_rdf_to_fuseki(file_path, fuseki_server, dataset_name, graph_uri=None):
    """
    Upload RDF file to Apache Fuseki triplestore.

    Args:
        file_path: Path to the RDF file (Turtle format)
        fuseki_server: Fuseki server URL (e.g., http://localhost:3030)
        dataset_name: Name of the dataset/graph in Fuseki
        graph_uri: Optional named graph URI. If None, uploads to default graph

    Returns:
        bool: True if successful, False otherwise
    """
    endpoint = f"{fuseki_server}/{dataset_name}/data"

    if graph_uri:
        endpoint += f"?graph={graph_uri}"

    try:
        with open(file_path, 'rb') as f:
            rdf_data = f.read()
    except FileNotFoundError:
        print(f"Error: File not found: {file_path}")
        return False

    headers = {
        'Content-Type': 'text/turtle; charset=utf-8'
    }

    print(f"Uploading {file_path} to Fuseki...")
    print(f"Endpoint: {endpoint}")
    print(f"File size: {len(rdf_data)} bytes")

    try:
        response = requests.post(
            endpoint,
            data=rdf_data,
            headers=headers,
            timeout=3600
        )

        if response.status_code in [200, 201, 204]:
            print("Successfully uploaded RDF data to Fuseki")
            return True
        else:
            print(f"Upload failed with status code: {response.status_code}")
            print(f"Response: {response.text}")
            return False

    except requests.exceptions.RequestException as e:
        print(f"Error uploading to Fuseki: {e}")
        return False


# Clear any partial data in the target dataset before uploading
_clear_endpoint = f"{fuseki_server}/{graph_name}/update"
print(f"Clearing dataset {graph_name} before upload...")
_clear_resp = requests.post(
    _clear_endpoint,
    data={'update': 'DROP ALL'},
    timeout=600,
)
if _clear_resp.status_code not in (200, 204):
    raise RuntimeError(
        f"Failed to clear dataset {graph_name}: "
        f"HTTP {_clear_resp.status_code} - {_clear_resp.text[:500]}"
    )
print("Dataset cleared.")

# DROP ALL removes ontology triples too; re-upload before loading trajectory data.
print("Re-uploading ontologies after DROP ALL...")
upload_ontologies(graph_name)

success = upload_rdf_to_fuseki(
    file_path=TRAJECTORY_DATA_TTL,
    fuseki_server=fuseki_server,
    dataset_name=graph_name
)

if success:
    print(f"\n{'='*60}")
    print("RDF DATA SUCCESSFULLY LOADED TO FUSEKI")
    print(f"{'='*60}")
    print(f"Dataset: {graph_name}")
    print(f"Fuseki Server: {fuseki_server}")
    print(f"Query endpoint: {fuseki_server}/{graph_name}/sparql")
    print("\nYou can now run enrichment queries (notebook 3)")
else:
    raise RuntimeError(
        f"Upload to Fuseki failed for dataset {graph_name}. "
        "Downstream notebooks cannot run against incomplete data."
    )


## Upload RDF Data to Fuseki

Upload the generated trajectory data to the Fuseki triplestore.